In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import h5py
import importlib

# Support opening notebooks from the root of the repository
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "src" / "observations.py").is_file():
    raise RuntimeError("Open the this notebook from the repo root or notebooks folder.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.observations import load_spectrum
import src.instrument as instrument

In [2]:
spectrum = load_spectrum(ROOT / "data" / "raw" / "observations" / "pg1048_all.dat")

forest = (spectrum.wavelength >= 1220.0) & (spectrum.wavelength <= 1380.0)

wavelength = spectrum.wavelength[forest]
valid = spectrum.valid[forest]

# NaNs break the plotly line plot, so we need to filter them out
# original spectrum is preserved in the `spectrum` variable

flux_plot = np.where(valid, spectrum.flux[forest], np.nan)
error_plot = np.where(valid, spectrum.error[forest], np.nan)

In [3]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=wavelength,
        y=flux_plot,
        customdata=error_plot,
        mode="lines",
        name="Normalized Flux",
        line=dict(color='#24476B', width=1),
        connectgaps=False,
        hovertemplate=(
            "Wavelength: %{x:.3f} Å"
            "<br>Normalized Flux: %{y:.3f}"
            "<br>Flux Error: %{customdata:.3f}"
            "<extra></extra>"
        ),
    )
)

fig.add_hline(y=1.0, line_dash="dash", line_color="gray")

fig.update_layout(
    title="COS Spectrum of PG1048 - candidate Lyman-alpha Forest Region",
    xaxis_title="Wavelength (Å)",
    yaxis_title="Normalized Flux",
    template="plotly_white",
    height=500,
    xaxis=dict(
        range=[1220, 1380],
        rangeslider=dict(visible=True, thickness=0.12)
    ),
)

fig.show(config={"scrollZoom": True})

In [4]:
overview = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.08,
)

# Reuse the flux trace from the first figure.
overview.add_trace(fig.data[0], row=1, col=1)

overview.add_trace(
    go.Scatter(
        x=wavelength,
        y=error_plot,
        mode="lines",
        name="Reported uncertainty",
        line=dict(color="#B45309", width=1),
        connectgaps=False,
        hovertemplate=(
            "Wavelength: %{x:.3f} Å"
            "<br>Reported error: %{y:.4f}"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

overview.add_hline(
    y=1.0, line_dash="dash", line_color="gray", row=1, col=1
)

overview.update_yaxes(title_text="Normalized flux", row=1, col=1)
overview.update_yaxes(
    title_text="Reported 1σ error",
    rangemode="tozero",
    row=2,
    col=1,
)
overview.update_xaxes(
    title_text="Observed wavelength [Å]", row=2, col=1
)
overview.update_xaxes(range=[1220.0, 1380.0])

overview.update_layout(
    title="pg1048_all.dat — spectrum and uncertainty",
    template="plotly_white",
    height=650,
    showlegend=False,
)

overview.show(config={"scrollZoom": True})

median_error = np.nanmedian(error_plot)
print(f"Median normalized uncertainty: {median_error:.4f}")
print(f"Approximate continuum S/N: {1.0 / median_error:.1f} per pixel")

Median normalized uncertainty: 0.0700
Approximate continuum S/N: 14.3 per pixel


In [5]:
C_KMS = 299792.458
LOS_LENGTH_KMS = 2500.0

usable_indices = np.flatnonzero(forest & spectrum.valid)

if usable_indices.size < 2:
    raise ValueError("Not enough usable pixels in the selected range.")

typical_spacing = np.median(
    np.diff(spectrum.wavelength[forest])
)

# Split at excluded rows or unusually large wavelength jumps.
breaks = (
    (np.diff(usable_indices) > 1)
    | (
        np.diff(spectrum.wavelength[usable_indices])
        > 3.0 * typical_spacing
    )
)

split_positions = np.flatnonzero(breaks) + 1
segments = np.split(usable_indices, split_positions)

rows = []

for number, indices in enumerate(segments, start=1):
    wave = spectrum.wavelength[indices]

    # Use pixel-center endpoints for a conservative coverage estimate.
    span_kms = C_KMS * np.log(wave[-1] / wave[0])
    full_los = int(np.floor(span_kms / LOS_LENGTH_KMS))

    rows.append({
        "segment": number,
        "start_A": wave[0],
        "end_A": wave[-1],
        "pixels": len(wave),
        "span_kms": span_kms,
        "full_LOS": full_los,
    })

coverage = pd.DataFrame(rows)

display(
    coverage.round({
        "start_A": 2,
        "end_A": 2,
        "span_kms": 1,
    })
)

print(f"Total full simulation-length chunks: {coverage['full_LOS'].sum()}")

,segment,start_A,end_A,pixels,span_kms,full_LOS
0,1,1220.02,1300.99,2711,19264.5,7
1,2,1306.91,1379.99,2447,16312.7,6


Total full simulation-length chunks: 13


In [6]:
lsf_path = (
    ROOT / "data" / "reference" / "cos_lsf"
    / "aa_LSFTable_G130M_1291_LP1_cn.dat"
)

lsf_wavelengths, lsf_kernels = instrument.load_cos_lsf(lsf_path)

print(f"Wavelength range: {lsf_wavelengths[0]:.0f}–{lsf_wavelengths[-1]:.0f} Å")
print(f"Kernel array shape: {lsf_kernels.shape}")

np.testing.assert_allclose(
    lsf_kernels.sum(axis=0),
    1.0,
    rtol=0.0,
    atol=1e-12,
)

Wavelength range: 1134–1430 Å
Kernel array shape: (321, 57)


In [7]:
TARGET_WAVELENGTH_A = 1300.0
DISPERSION_A_PER_PIXEL = 0.00997  # Nominal G130M dispersion
GAUSSIAN_SIGMA_KMS = 7.96
C_KMS = 299792.458

# Select the closest tabulated wavelength.
index = int(np.argmin(np.abs(lsf_wavelengths - TARGET_WAVELENGTH_A)))
lsf_wave = float(lsf_wavelengths[index])
lsf_kernel = lsf_kernels[:, index]

# Convert native LSF pixel offsets to approximate velocity offsets.
pixel_offsets = np.arange(lsf_kernel.size) - lsf_kernel.size // 2
dv_lsf = C_KMS * DISPERSION_A_PER_PIXEL / lsf_wave
lsf_velocity = pixel_offsets * dv_lsf

# Evaluate the previous Gaussian approximation on the same grid.
gaussian_kernel = np.exp(
    -0.5 * (lsf_velocity / GAUSSIAN_SIGMA_KMS) ** 2
)
gaussian_kernel /= gaussian_kernel.sum()

lsf_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Line core", "Wings — logarithmic scale"],
)

profiles = [
    ("COS LP1", lsf_kernel, "#24476B"),
    ("Gaussian σ = 7.96 km/s", gaussian_kernel, "#B45309"),
]

for label, kernel, color in profiles:
    density = kernel / dv_lsf

    for column in (1, 2):
        # Zero values cannot be displayed on a logarithmic axis.
        plotted = density if column == 1 else np.where(
            density > 0.0, density, np.nan
        )

        lsf_fig.add_trace(
            go.Scatter(
                x=lsf_velocity,
                y=plotted,
                mode="lines",
                name=label,
                legendgroup=label,
                showlegend=(column == 1),
                line=dict(color=color, width=2),
                connectgaps=False,
            ),
            row=1,
            col=column,
        )

lsf_fig.update_xaxes(title_text="Velocity offset [km/s]")
lsf_fig.update_xaxes(range=[-60, 60], row=1, col=1)
lsf_fig.update_xaxes(
    range=[lsf_velocity[0], lsf_velocity[-1]], row=1, col=2
)
lsf_fig.update_yaxes(title_text="LSF density [(km/s)⁻¹]")
lsf_fig.update_yaxes(type="log", range=[-7, -1], row=1, col=2)

lsf_fig.update_layout(
    title=f"LP1 / assumed G130M 1291 — LSF at {lsf_wave:.0f} Å",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", y=-0.25),
)
lsf_fig.show(config={"scrollZoom": True})

print(f"Native LSF sampling: {dv_lsf:.3f} km/s per pixel")

Native LSF sampling: 2.299 km/s per pixel


In [8]:
importlib.reload(instrument)

dv_sim = 2500.0 / 2499
kernel_sim = instrument.resample_lsf(
    lsf_kernel,
    dv_in=dv_lsf,
    dv_out=dv_sim,
)

print(f"Simulation sampling: {dv_sim:.6f} km/s per pixel")
print(f"Kernel length: {kernel_sim.size}")
print(f"Kernel sum: {kernel_sim.sum():.12f}")

np.testing.assert_allclose(
    kernel_sim.sum(), 1.0, rtol=0.0, atol=1e-12
)
assert np.all(kernel_sim >= 0.0)
assert kernel_sim.size % 2 == 1

Simulation sampling: 1.000400 km/s per pixel
Kernel length: 739
Kernel sum: 1.000000000000


In [9]:
importlib.reload(instrument)

# Two diagnostic inputs: a flat continuum and a varying spectrum.
probe = np.vstack([
    np.ones(2499),
    np.linspace(0.0, 1.0, 2499),
])

smoothed, dv_out = instrument.apply_cos_lsf(
    probe,
    dv_sim=dv_sim,
    kernel=kernel_sim,
)

assert smoothed.shape == probe.shape
assert dv_out == dv_sim

# A flat continuum should remain flat.
np.testing.assert_allclose(
    smoothed[0], probe[0], rtol=0.0, atol=1e-12
)

# Each sightline should retain its own mean flux.
np.testing.assert_allclose(
    smoothed.mean(axis=-1),
    probe.mean(axis=-1),
    rtol=0.0,
    atol=1e-12,
)

print("Convolution checks passed.")

Convolution checks passed.


In [10]:
MOCK_MODEL = "EX0"
MOCK_LOS = 0

calibration_path = (
    ROOT / "outputs" / "uvb_mean_flux" / "calibration"
    / "z0_b15_seed42.json"
)

with calibration_path.open() as handle:
    calibration = json.load(handle)

if calibration["redshift"] != 0.0:
    raise ValueError("This example assumes z = 0")

tau_scale = float(calibration["results"][MOCK_MODEL]["tau_scale"])

if not np.isfinite(tau_scale) or tau_scale <= 0.0:
    raise ValueError("Expected a positive, finite optical-depth scale")

mock_path = ROOT / "data" / "raw" / f"{MOCK_MODEL}_spectra.hdf5"

# Read only the selected sightline.
with h5py.File(mock_path, "r") as handle:
    tau_mock = np.asarray(
        handle[calibration["tau_key"]][MOCK_LOS, :],
        dtype=np.float64,
    )

if tau_mock.ndim != 1 or tau_mock.size < 2:
    raise ValueError("Expected one sightline with at least two pixels")

if not np.all(np.isfinite(tau_mock)) or np.any(tau_mock < 0.0):
    raise ValueError("Optical depth must be finite and nonnegative")

flux_native = np.exp(-tau_scale * tau_mock)
dv_mock = 2500.0 / flux_native.size
velocity_mock = np.arange(flux_native.size) * dv_mock

# Prepare the LSF using this sightline's actual pixel spacing.
kernel_mock = instrument.resample_lsf(
    lsf_kernel,
    dv_in=dv_lsf,
    dv_out=dv_mock,
)

flux_cos, _ = instrument.apply_cos_lsf(
    flux_native,
    dv_sim=dv_mock,
    kernel=kernel_mock,
)

flux_gaussian, _ = instrument.apply_cos_gaussian(
    flux_native,
    dv_sim=dv_mock,
    sigma_kms=7.96,
    truncate=8.0,
)

np.testing.assert_allclose(
    [flux_cos.mean(), flux_gaussian.mean()],
    flux_native.mean(),
    rtol=0.0,
    atol=1e-12,
)

mock_fig = go.Figure()

for label, values, color, dash in [
    ("Native mock", flux_native, "#808080", "dot"),
    ("Gaussian σ = 7.96 km/s", flux_gaussian, "#B45309", "dash"),
    ("COS LP1", flux_cos, "#24476B", "solid"),
]:
    mock_fig.add_trace(
        go.Scatter(
            x=velocity_mock,
            y=values,
            mode="lines",
            name=label,
            line=dict(color=color, dash=dash, width=1.5),
            hovertemplate=(
                "Velocity: %{x:.1f} km/s"
                "<br>Flux: %{y:.4f}"
                "<extra>%{fullData.name}</extra>"
            ),
        )
    )

mock_fig.update_layout(
    title=(
        f"{MOCK_MODEL}, sightline {MOCK_LOS} — mean-flux matched"
        f"<br><sup>LP1; assumed G130M/1291; "
        f"single LSF profile at {lsf_wave:.0f} Å</sup>"
    ),
    xaxis_title="Velocity along the simulated sightline [km/s]",
    yaxis_title="Normalized flux",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.01),
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.1)),
)

mock_fig.show(config={"scrollZoom": True})
print(f"Mean-flux checks passed. Native mean: {flux_native.mean():.6f}")

Mean-flux checks passed. Native mean: 0.989885
